# Lesson 02 - మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ అన్వేషణ

**మైక్రోసాఫ్ట్ ఏజంట్ ఫ్రేమ్‌వర్క్ (MAF)** అనేది AI ఏజెంట్లను నిర్మించడానికి ఒక ఏకీకృత ఫ్రేమ్‌వర్క్. ఇది నాలుగు ప్రధాన నిర్మాణ 블ాక్స్‌తో శుభ్రమైన, సంగ్రహించగల ఆర్కిటెక్చర్‌ను అందిస్తుంది:

- **క్లయింట్** – AI మోడల్ ఎండ్‌పాయింట్‌కి కనెక్ట్ అయ్యి కమ్యూనికేషన్‌ను నిర్వహిస్తుంది
- **ఏజెంట్** – సూచనలు మరియు టూల్ నిర్వచనాలతో క్లయింట్‌ని చుట్టుకుంటుంది
- **టూల్స్** – మోడల్ కాల్ చేయగల కస్టమ్ ఫంక్షన్లతో ఏజెంట్ సామర్థ్యాలను విస్తరించును
- **సెషన్** – బహుళ-టర్న్ ఇంటరాక్షన్ల కోసం సంభాషణ చరిత్రను నిల్వచేస్తుంది

ఈ పాఠంలో, మేము ఈ కాన్సెప్ట్లను ఉపయోగించి గమ్యస్థానం అందుబాటును తనిఖీ చేసే **ప్రయాణ బుకింగ్ ఏజెంట్** ను నిర్మించబోతున్నాము.


## సెటప్


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ఏజెంట్ ఫ్రేమ్‌వర్క్ ఆర్కిటెక్చర్‌ను అర్థం చేసుకోవడం

మైക്രోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఒక పొరల ఆర్కిటెక్చర్‌ను అనుసరిస్తుంది:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **క్లయింట్** – ఒక `AzureAIProjectAgentProvider` Azure OpenAI డిప్లాయ్‌మెంట్‌కు కనెక్ట్ అవుతుంది. ఇది ప్రామాణీకరణ, అభ్యర్థన ఫార్మాటింగ్ మరియు ప్రతిస్పందన పార్సింగ్‌ను నిర్వహిస్తుంది.
2. **ఏజెంట్** – `provider.create_agent()` ద్వారా క్లయింట్ నుండి సృష్టించబడిన ఏజెంట్, ఇది మోడల్ యాక్సెస్‌ను సూచనలు (సిస్టమ్ ప్రాంప్ట్) మరియు టూల్స్‌తో కలిపి ఉంటుంది.
3. **టూల్స్** – `@tool` తో డెకరేట్ చేయబడిన Python ఫంక్షన్లు, ఏజెంట్ వీటి ద్వారా చర్యలను నిర్వర్తించడానికి లేదా డేటాను పొందడానికి కాల్ చేయగలదు.
4. **సెషన్** – ఒక `AgentSession` ఆబ్జెక్ట్ (agent.create_session() ద్వారా సృష్టించబడినది) ఇది సంభాషణ చరిత్రను నిల్వచేసి, ఏజెంట్ గత సంభాషణ సందర్భాన్ని గుర్తుంచుకునే బహుళ-తిరుగుడు సంభాషణను సాధ్యమయ్యేలా చేస్తుంది.

ప్రతి పొరను దశలవారీగా నిర్మిద్దాం.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool డెకొరేటర్‌తో టూల్స్ జోడించడం

టూల్స్ ఏజెంట్లు టెక్స్ట్ ఉత్పత్తిని మించి చర్యలు తీసుకోవడానికి అవకాశం ఇస్తాయి. `@tool` డెకొరేటర్ సాధారణ Python ఫంక్షన్‌ను ఏజెంట్ పిలవగలిగే విధంగా మార్చుతుంది.

ప్రధాన బిందువులు:  
- మోడల్ ప్రతి పారామీటర్ని అర్థం చేసుకునేందుకు `Annotated[type, "description"]` ఉపయోగించండి.  
- డోక్యు స్ట్రింగ్ టూల్ వివరణగా మారుతుంది, ఇది మోడల్ చూస్తుంది.  
- `approval_mode="never_require"` అంటే యూజర్ నిర్ధారణ లేకుండా టూల్ ఆటోమేటిక్‌గా నడుస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## టూల్స్‌తో ఏజెంట్ సృష్టించడం

ఇప్పుడు మనం కస్టమర్, మార్గదర్శకాలు, మరియు టూల్స్‌ను ఒక ఏజెంట్‌గా కలుపుతాము. `instructions` సిస్టమ్ ప్రాంప్ట్‌గా పనిచేస్తాయి — అవి ఏజెంట్ యొక్క వ్యక్తిత్వం మరియు ప్రవర్తనను నిర్వచిస్తాయి.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## సెషన్లతో బహు-తిరుగుబాటు సంభాషణలు

ఒక `AgentSession` (`agent.create_session()` ద్వారా సృష్టించబడిన) సంభాషణలోని అన్ని సందేశాలను ట్రాక్ చేస్తుంది. ప్రతి `agent.run()` కాల్‌కు అదే సెషన్‌ను పంపడం ద్వారా, ఏజెంట్ పూర్తి సంభాషణ చరిత్రను పొందగలడు మరియు ముందటి సందేశాలను తిరిగి సూచించగలడు.

ప్రతి తిప్పడిలో మా అందుబాటు తనిఖీదారుని కాల్ చేయడానికి ఏజెంట్ అందుబాటు తనిఖీదారుని ఉపయోగించడానికి `tools=[check_destination_availability]` ను పంపిస్తున్నాము.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## సారాంశం

ఈ పాఠంలో మీరు Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్ యొక్క నాలుగు స్తంభాలను అన్వేషించారు:

| కాన్సెప్ట్ | మీరు నేర్చుకున్నది |
|---------|------------------|
| **క్లయింట్** | `AzureAIProjectAgentProvider` క్రెడెన్షియల్ ఆధారిత ప్రమాణీకరణతో Azure OpenAIకి కనెక్ట్ అవుతుంది |
| **ఏజెంట్** | `provider.create_agent()` మోడల్ కనెక్షన్‌ను నిబంధనలు మరియు ఒక పేరుతో బండిల్ చేస్తుంది |
| **టూల్స్** | `@tool` డెకొరేటర్ ఏజెంట్ పిలవడానికి Python ఫంక్షన్లను ప్రదర్శిస్తుంది |
| **సెషన్** | `agent.create_session()` బహుళ టర్న్లలో సంభాషణ చరిత్రను నిర్వహిస్తుంది |

ఈ నిర్మాణం కలిసి సహజ సంభాషణలు నిర్వహించగల ఏజెంట్లను, బాహ్య ఫంక్షన్లు పిలవగలగడం, మరియు సందర్భాన్ని నిర్వహించగలగడం — తదుపరి పాఠాలలో మరింత అభివృద్ధి చెందిన ఏజెంటిక్ నమూనాలకు మూలస్తంభంగా ఉంటుంది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత**:
ఈ డాక్యుమెంట్ AI అనువాద సేవ అయిన [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వం కోసం కృషి చేస్తున్నప్పటికీ, స్వయంసేవిత అనువాదాల్లో తప్పులేమో లేదా అపసత్యాలు ఉండొచ్చు. మూల డాక్యుమెంట్ దాని మాతృభాషలోని అధికారిక వనరు గా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, నిపుణుల మానవ అనువాదాన్ని సూచిస్తాము. ఈ అనువాదం ఉపయోగం వల్ల వచ్చిన ఏదైనా అవగాహన లోపాలు లేదా తప్పు అర్థాలు పొందడంపై మేము బాధ్యులు కారు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
